## 0. Setup — Run This First

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# ── Simulate the Dataset ──
rng = np.random.default_rng(42)
n = 200
age        = rng.integers(18, 70, n)
income     = rng.integers(15, 140, n)
spend_base = 100 - 0.4 * income + rng.normal(0, 18, n)
spend      = np.clip(spend_base, 1, 100).astype(int)
gender     = rng.integers(0, 2, n)
df = pd.DataFrame({'Gender': gender, 'Age': age, 'Income': income, 'SpendingScore': spend})

print(f"\nDataset shape: {df.shape}")
df.head()


---
## Section 1 — Exploring the Data

### Exercise 1.1 — Inspect the Data

Use `.info()` and `.describe()` to understand the dataset.
Then print the number of missing values per column.


In [ ]:
# ── Your code here ──


In [ ]:
print("Missing values per column:")
print(df.isnull().sum())


### Exercise 1.2 — Visualise Pairwise Relationships

Create a pairplot to visually inspect natural groupings before any modelling. Use `hue='Gender'`.

This is a function from Seaborn that generates a grid of plots showing relationships between all pairs of numerical columns in your DataFrame df.

What appears in the plot?

Off-diagonal plots → scatter plots between every pair of features

Diagonal plots → distribution of each individual feature

In [ ]:
# ── Your code here ──


**Answer 1.2**

In [ ]:
sns.pairplot(df, hue='Gender', palette={0: 'steelblue', 1: 'salmon'}, plot_kws={'alpha': 0.6}, diag_kind='kde')
plt.suptitle('Pairwise Feature Relationships', y=1.02, fontsize=14)
plt.show()



### Exercise 2.1 — Scale the Features

1. Define your features to include `Age`, `Income`, and `SpendingScore`.
2. Apply `StandardScaler` so every feature has mean = 0 and standard deviation = 1.
3. Print the scaled means and standard deviations to verify.

In [ ]:
# ── Your code here ──


**Answer 2.1**

In [ ]:
features = ['Age', 'Income', 'SpendingScore']
X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

print("After scaling — mean and std per feature:")
print(X_scaled_df.describe().loc[['mean', 'std']].round(4))


---
## 3. K-Means Clustering

### Exercise 3.1 — The Elbow Method and Silhouette Score

Choosing $K$ is the central challenge.
1. Loop through $K$ from 2 to 10.
2. For each, fit a `KMeans` model and calculate its `inertia_` and `silhouette_score`.
3. Plot both metrics side-by-side to find the optimal $K$.


In [ ]:
# ── Your code here ──


**Answer 3.1**

In [ ]:
inertia_values = []
silhouette_values = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertia_values.append(km.inertia_)
    silhouette_values.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertia_values, marker='o', color='steelblue',
             linestyle='--', markerfacecolor='crimson', markersize=8)
axes[0].set_title('Elbow Method — Inertia vs K', fontsize=13)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_xticks(list(k_range))
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_range, silhouette_values, marker='s', color='darkorange',
             linestyle='--', markerfacecolor='purple', markersize=8)
axes[1].set_title('Silhouette Score vs K', fontsize=13)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_xticks(list(k_range))
axes[1].grid(True, alpha=0.3)

plt.suptitle('Choosing the Optimal K', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

best_k = k_range.start + np.argmax(silhouette_values)
print(f"Best K by silhouette score: {best_k}  (score = {max(silhouette_values):.4f})")



### Exercise 3.2 — Fitting K-Means with the Optimal K

Fit a final `KMeans` model using the `OPTIMAL_K` value found above. Print out the final inertia, silhouette score, and the size of each cluster.


In [ ]:
# ── Your code here ──


**Answer 3.2**

In [ ]:
OPTIMAL_K = best_k

kmeans = KMeans(n_clusters=OPTIMAL_K, n_init=10, random_state=42)
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

print(f"K-Means fitted with K={OPTIMAL_K}")
print(f"Final inertia : {kmeans.inertia_:.2f}")
print(f"Silhouette    : {silhouette_score(X_scaled, df['KMeans_Cluster']):.4f}\n")
print("Cluster sizes:")
print(df['KMeans_Cluster'].value_counts().sort_index())


### Exercise 3.3 — Visualising the Clusters (Income vs Spending Score)

Let's focus visually on the 2 features that showed the most distinct groupings: `Income` and `SpendingScore`.
1. Refit a K-Means model on just these two scaled features.
2. Inverse-transform the centroids so they map back to original data scales.
3. Plot a scatter plot of the clusters and mark the centroids heavily.


In [ ]:
# ── Your code here ──


**Answer 3.3**

In [ ]:
palette = sns.color_palette('Set2', OPTIMAL_K)

X2 = df[['Income', 'SpendingScore']].copy()
scaler2 = StandardScaler()
X2_scaled = scaler2.fit_transform(X2)
km2 = KMeans(n_clusters=OPTIMAL_K, n_init=10, random_state=42)
labels_2d = km2.fit_predict(X2_scaled)
centroids_orig = scaler2.inverse_transform(km2.cluster_centers_)

plt.figure(figsize=(9, 6))
for i in range(OPTIMAL_K):
    mask = labels_2d == i
    plt.scatter(df['Income'][mask], df['SpendingScore'][mask],
                color=palette[i], label=f'Cluster {i}', s=70, alpha=0.75, edgecolors='white')

plt.scatter(centroids_orig[:, 0], centroids_orig[:, 1],
            s=280, c='black', marker='X', zorder=5, label='Centroids')

plt.title(f'K-Means Clusters (K={OPTIMAL_K}) — Income vs Spending Score', fontsize=13)
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1–100)')
plt.legend()
plt.tight_layout()
plt.show()
